# Intro to DIZON: the field experiment, the chemistry, the decision

_The site that this whole tutorial series is built on, before any code._


This is a **part0** notebook: standalone background, no prerequisites, safe to read in any order. Nothing here runs the big model. We are just setting the scene.

The reactive-transport modelling starts in the part1 sequence, beginning with [`part1_01_build_model`](../part1_01_build_model/dizon_build_model.ipynb). If you want the miniature version of this chemistry running in seconds, the forthcoming `part0_02_intro_to_mf6rtm` pyrite-column notebook is the place to go.

By the end of this notebook you should be able to answer three questions:

1. **What was measured in the field?** (the DIZON experiment at Someren)
2. **What is the chemistry that makes it interesting?** (pyrite oxidation, and why temperature matters)
3. **What decision are we using it to support?** (how much sulfate the supply well will carry, and how sure we are)


### Admin

There is very little code in this notebook. We load one CSV of field data near the end to show that the measured breakthrough is real, and we build one timeline figure that the later notebooks reuse. No model is run here, so there is no expensive wait: full DIZON runs cost roughly **6 minutes each** (on a MacBook; 10-15 min on Windows), and that cost is the reason this curriculum exists. We will keep reminding you of it.

The next cell imports what we need. The field data lives in the shared `data/` directory at the repository root.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

plt.rcParams["font.size"] = 10

# field data + GIS inputs live in the repository-root data/ directory (two levels up)
data_dir = os.path.join("..", "..", "data")
assert os.path.isdir(data_dir), "could not find the data/ directory at ../../data"


## The site and the experiment

This tutorial series is built on the **DIZON** study — _Deep well Injection in Zuid-Oost Nederland_ — a field Aquifer Storage and Recovery (ASR) experiment at **Someren**, in the south-east of the Netherlands, and its accompanying reactive transport model. The original study is [Prommer and Stuyfzand (2005)](https://pubs.acs.org/doi/10.1021/es0486768) (_Environmental Science & Technology_ **39**, 2200–2209).

The setup, in one sentence: **pre-treated, oxic surface water was injected about 300 m deep into an anoxic, pyrite-rich aquifer**, and the resulting chemistry was monitored over roughly **854 days** at a set of observation wells at different distances and depths from the injection point.

What makes this a _reactive_ transport problem rather than a plain tracer test is what happens when oxygenated water meets pyrite. The injectate is chemically out of equilibrium with the aquifer, and the aquifer reacts. The field dataset captures that reaction front moving and evolving through space and time — which is exactly the kind of signal a reactive transport model is built to reproduce, and exactly the kind of signal whose _future_ behaviour we will eventually want to forecast under uncertainty.

## The chemistry: pyrite oxidation, and why temperature matters

The aquifer contains **pyrite (FeS₂)** and a minor amount of sediment-bound organic matter. The injected water carries dissolved **oxygen (O₂)** and **nitrate (NO₃⁻)**. Where they meet, pyrite is oxidised. Schematically, oxygen and nitrate are consumed while sulfate and acidity are produced, with ferric iron precipitating out:

$$\text{FeS}_2 + \text{O}_2 / \text{NO}_3^- \;\longrightarrow\; \text{SO}_4^{2-} + \text{H}^+ + \text{Fe(OH)}_3 \downarrow$$

Pyrite oxidation is the **dominant redox control** on water quality in this system. Organic matter is a minor contributor — a finding of the source study that we lean on later when we decide what to put in, and leave out of, the uncertain parameter set.

The headline result of Prommer and Stuyfzand (2005) is that the **reaction rates are strongly temperature-dependent**, and that you cannot reproduce the observed seasonal redox patterns without that dependence. The injected water has a seasonally varying temperature, and:

- **cold injectate** → slower reaction → oxygen and nitrate penetrate _further_ into the aquifer before being consumed;
- **warm injectate** → faster reaction → oxidants are stripped out _close_ to the injection well.

So temperature is not a side detail — it controls _where_ the redox front sits and _how fast_ it moves. In the model, temperature is carried as a retarded heat "tracer" (heat exchanges with the aquifer matrix, so the thermal signal lags the solute signal), and that simulated temperature field feeds straight back into the reaction rates. Flow, heat, and geochemistry are coupled; you cannot interpret one without the others.

We meet this exact reaction network twice more in the series, at two very different scales:

- in miniature, on a 1D **pyrite column** that runs in seconds, in the forthcoming `part0_02_intro_to_mf6rtm` notebook — the place to build redox-front intuition (and to feel the warm-vs-cold injectate effect) cheaply;
- at full field scale, in the 3D DIZON model assembled in [`part1_01_build_model`](../part1_01_build_model/dizon_build_model.ipynb), where one run is the ~6-minute model.

## The decision, recast

The original study was a scientific investigation. We have recast it as a **decision-support** problem, which is what the rest of the curriculum is about.

The recast question is deliberately simple to state:

> We are injecting oxic, warmer, pre-treated water into a cooler, anoxic, pyrite-rich aquifer. That oxidises pyrite and releases **sulfate**. We want to extract water from a **supply well** for drinking water. **How much sulfate will the supplied water carry, and how sure are we?**

This is a _design_ question, not a pass/fail one. More sulfate means more treatment cost, so the operator sizes treatment capacity to the high end of what the water might carry — in practice the **95th percentile of the peak sulfate concentration** at the supply well over the period it operates. The forecast we carry through every later notebook is therefore the full **distribution** of that peak — its median and its P95 — not a single number.

Drinking-water compliance is not the binding constraint here: the EU sulfate standard is **250 mg/L** and the real DIZON supply water peaked around **110 mg/L**, comfortably under it — so cost, not compliance, drives this decision. _Easy, right?_ It is not, because the answer depends on uncertain subsurface properties, which is where the uncertainty-quantification machinery comes in.


### The three wells

The simulation has three pumping wells. We use descriptive names in prose and give the code IDs once, here:

- **injection well** (`wellin`) — injects oxic water for the whole simulation;
- **flush well** (`wellout`) — extracts during the early _history period_;
- **supply well** (`wellopt`) — extracts drinking water later, during the _supply period_; this is where the forecast lives.

(The "opt" in `wellopt` is just a code label — think of it as the **supply** well throughout.)

## The timeline

Three dates organise the whole curriculum, so it is worth a picture. The tutorial runs on a **728-day** timeline:

- **history period** — days **0–252**: injection plus flush-well operation; this is the window in which monitoring data exist and may be used to condition the model;
- **decision date** — day **252**: the operating decision must be committed _here_, before the supply well switches on;
- **supply period** — days **308–728**: the supply well extracts drinking water; the forecast (peak sulfate) lives entirely in this window.

The gap between the decision date (252) and the start of supply (308) is deliberate: real decisions are made with **lead time**, not on the morning the pump starts. We will lean on that gap repeatedly.

The helper below draws this timeline. It is intentionally small and self-contained so that later notebooks can reuse it (the same history / decision-date / supply bands show up whenever we plot a forecast). Run it to see the figure:

In [ ]:
# canonical curriculum dates (days since start of injection)
HISTORY_END   = 252   # end of the history period (also the decision date)
DECISION_DATE = 252
SUPPLY_START  = 308
SUPPLY_END    = 728


def plot_timeline(ax=None):
    """Draw the DIZON tutorial timeline: history period, decision date, supply period.

    Pass an existing axis to overlay the bands on a forecast plot; otherwise a new
    figure is created. Returns the axis so callers can keep drawing on it.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 2.2))

    ax.add_patch(Rectangle((0, 0.0), HISTORY_END, 1.0, alpha=0.30,
                           color="tab:blue", label="history period (0-252 d)"))
    ax.add_patch(Rectangle((SUPPLY_START, 0.0), SUPPLY_END - SUPPLY_START, 1.0,
                           alpha=0.30, color="tab:green",
                           label="supply period (308-728 d)"))

    ax.axvline(DECISION_DATE, color="k", lw=2, ls="--")
    ax.annotate("decision date\n(day 252)", xy=(DECISION_DATE, 1.05),
                ha="center", va="bottom", fontsize=9)

    # the lead-time gap between deciding and the supply well switching on
    ax.annotate("", xy=(SUPPLY_START, 0.5), xytext=(DECISION_DATE, 0.5),
                arrowprops=dict(arrowstyle="->", color="0.3"))
    ax.text((DECISION_DATE + SUPPLY_START) / 2, 0.58, "lead-time gap",
            ha="center", va="bottom", fontsize=8, color="0.3")

    ax.set_xlim(0, SUPPLY_END + 20)
    ax.set_ylim(0, 1.4)
    ax.set_yticks([])
    ax.set_xlabel("time since start of injection (days)")
    ax.legend(loc="lower right", fontsize=8, framealpha=0.9)
    ax.set_title("DIZON tutorial timeline")
    return ax


plot_timeline()
plt.tight_layout()
plt.show()


## The measured breakthrough is real

In a rare win for a tutorial, we actually have **measured field data**. It lives in `data/obs_chem_cleaned.csv` and covers several monitoring sites, each potentially with multiple screened intervals at different depths, for a range of chemical species over the full experiment.

A word of warning before we lean on it: for almost the entire core sequence we do **not** history-match against this measured data. Instead we use a **synthetic truth** — a single realisation pulled from the prior ensemble, deliberately chosen so the decision is genuinely interesting, and then declared to be "the system". The measured data appears in exactly two places: here, as evidence the breakthrough is real, and in the optional real-data capstone. So this is a credibility check, not the truth.

Load the file and look at its shape:

In [ ]:
meas = pd.read_csv(os.path.join(data_dir, "obs_chem_cleaned.csv"))
meas["time"]  = pd.to_numeric(meas["time"],  errors="coerce")
meas["value"] = pd.to_numeric(meas["value"], errors="coerce")
meas.head()


The columns tell the story: **`obsid`** names the monitoring site and screened interval (e.g. `wp1-f3` is site `wp1`, filter depth `f3`); **`variable`** is the chemical species or property (concentrations in mol/L); **`time`** is days since injection started; and **`x`, `y`, `layer`** locate the screen. We unpack this `wp1-f3` naming scheme properly in the observation notebook — here we just want to see the breakthrough.

Look at how many sites and species are on offer:

In [ ]:
print("monitoring sites :", sorted(meas.obsid.unique()))
print()
print("species / props  :", sorted(meas.variable.dropna().unique()))


That is a lot of sites and a lot of species — which can feel daunting if you come from a flow-modelling background used to heads and fluxes and not much else. The core sequence deliberately conditions on only a handful: **SO₄** (the forecast species), **O₂** and **NO₃** (the oxidant-consumption signal — oxygen appears in the file as `O0`, dissolved O₂), **pH** (buffering), and **temperature** (a heat tracer that helps pin down velocities). The major cations (Ca, Mg, Na, K, Fe) are deliberately **held back** — we return to them as a dataworth question ("would measuring these have helped?") later on.

Let's plot the measured **sulfate** breakthrough at three sites, in the file's native mol/L units. The dashed line marks the decision date (day 252):

In [ ]:
sites = ["wp1-f3", "wp3-f3", "wp4-f5"]   # near-to-far from the injection well

fig, axs = plt.subplots(1, len(sites), figsize=(10, 2.8), sharey=True)
for ax, oid in zip(axs, sites):
    sub = meas[(meas.obsid == oid) &
               (meas.variable == "SO4") &
               meas.value.notna()].sort_values("time")
    ax.plot(sub.time, sub.value, marker="o", ms=3, lw=1, color="tab:red")
    ax.axvline(DECISION_DATE, color="0.5", ls="--", lw=1)
    ax.set_title(oid)
    ax.set_xlabel("day")
axs[0].set_ylabel("SO4 (mol/L)")
fig.suptitle("Measured sulfate breakthrough (DIZON field data)")
fig.tight_layout()
plt.show()


Sulfate clearly rises through the experiment as pyrite oxidation proceeds — the breakthrough is real, and a model that hopes to forecast supply-well sulfate has something concrete to be credible against. We compare the _built_ model against this data in [`part1_01_build_model`](../part1_01_build_model/dizon_build_model.ipynb) (credibility, not conditioning).

## One honest note on provenance

The tutorial model is a **modified, decision-support recast** of the published study, not a validated reproduction of it. The most visible difference: the field experiment ran for about **854 days**, while our tutorial timeline is **728 days**. That abstraction is acknowledged here and not reconciled — the chemistry is faithful to the source study, but the geometry, schedule, and well operations have been adapted to make a clean decision-support case. Treat the numbers as a teaching device, not as the published DIZON result.

## Where to go next

You now have the site, the chemistry, and the decision question. The natural next steps:

- **the chemistry in miniature** — the forthcoming `part0_02_intro_to_mf6rtm` pyrite-column notebook: the same pyrite reaction network on a 1D column that runs in seconds, to learn the mf6rtm API and the warm-vs-cold redox-front behaviour;
- **why we need uncertainty quantification at all** — [`part0_03_uq_for_rtm`](../part0_03_uq_for_rtm/uq_for_rtm.ipynb);
- **the full DIZON build** — [`part1_01_build_model`](../part1_01_build_model/dizon_build_model.ipynb), where one run is the ~6-minute model and the cost that motivates everything becomes real.